# Day 2 — Hands-On Lab 1: Storage Credentials, External Locations & Lakeflow Connect

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 2 — Lakeflow Connect + Storage Credentials & External Locations |
| **Source** | Your own Supabase Postgres project (`orders`, `order_items`) + your own ADLS storage account |
| **Duration** | 60 minutes |
| **Output** | A working Storage Credential + External Location, and a live Lakeflow Connect CDC pipeline querying Postgres via a cursor column (`updated_at`) |

### Learning Objectives
- Create your own Storage Credential backed by an Azure Managed Identity
- Create your own External Location and verify access with zero hardcoded keys
- Set up `orders`/`order_items` for CDC in Supabase — trigger, `REPLICA IDENTITY FULL`, publication — and load data via CSV import
- Create your own PostgreSQL connection to your own Supabase project
- Configure a Lakeflow Connect ingestion pipeline (query/cursor-based capture, cursor column `updated_at`, history tracking Off/SCD1, no schedule — triggered manually) for `orders` and `order_items`
- Prove CDC: change a row in Supabase, re-run the pipeline, confirm only the change syncs
- Prove the limit of cursor-based capture: hard-delete a row and confirm the pipeline does **NOT** detect it — the row stays in Bronze

---
**Instructions:** Run each cell with **Shift + Enter**. Phases A–C0 are mostly Azure Portal / Databricks UI / Supabase SQL Editor steps — read carefully, screenshot as you go for your submission. Phases C1–D are notebook cells.

---
## Phase A — Create Your Own Storage Credential

**Goal:** Register a Storage Credential in Unity Catalog backed by an Azure Managed Identity — same pattern as the real `ecomprojectscredentials` shown in ILT 2, but naming everything after yourself.

### A1 — Find (or create) your Access Connector

```
Azure Portal → search bar → "Access Connectors for Azure Databricks"
  → find the Access Connector linked to your Databricks workspace
  → Settings → Properties → copy the Resource ID
    (looks like: /subscriptions/.../resourceGroups/.../providers/Microsoft.Databricks/accessConnectors/...)

If none exists:
  Azure Portal → Create a resource → search "Azure Databricks Access Connector"
  → create it in the same Resource Group as your Databricks workspace
```

### A2 — Confirm the Access Connector has RBAC on your storage account

```
Azure Portal → your storage account (from Day 1, e.g. globalmart<yourname>)
  → Access Control (IAM) → Role assignments
  → confirm the Access Connector has: Storage Blob Data Contributor

If missing:
  → Add role assignment → Storage Blob Data Contributor
  → Assign access to: Managed Identity → select your Access Connector
```

### A3 — Create the Storage Credential in Databricks

```
Databricks → Catalog icon (left sidebar) → External Data → Storage Credentials
  → Create credential

  Credential type   : Azure Managed Identity
  Credential name   : <yourname>_credential        e.g. virinchy_credential
  Access Connector ID: (paste the Resource ID from A1)
  Comment           : Managed identity for my Day 1 storage account
  → Create
```

**Verify:** the credential appears in the list with status **Active**.

---
## Phase B — Create Your Own External Location

**Goal:** Map your ADLS container to the Storage Credential from Phase A — mirrors the real `gbmart-ext-loc` shown in ILT 2.

### B1 — Create the External Location

```
Databricks → Catalog icon → External Data → External Locations
  → Create location

  External location name : <yourname>_external        e.g. virinchy_external
  URL                     : abfss://<your-container>@<your-storage-account>.dfs.core.windows.net/
  Storage credential      : <yourname>_credential      (from Phase A)
  Comment                 : External location for my Day 1 storage account
  → Create
```

### B2 — Test the connection

```
Click "Test connection" on the External Location you just created.
Expected: ✅ Connection test succeeded — lists files under your container.

If you see a permissions error → go back to A2 and re-check the RBAC role assignment.
```

Fill in your own values below, then run the cells to verify access **with zero hardcoded keys**.

In [ ]:
# ─── B3: Verify access — NO spark.conf.set, NO storage key ───────────────────
# Replace with your own container / storage account from Day 1

your_container       = "YOUR_CONTAINER_NAME"               # ← your container name
your_storage_account = "YOUR_STORAGE_ACCOUNT_NAME"         # ← your storage account from Day 1

base_path = f"abfss://{your_container}@{your_storage_account}.dfs.core.windows.net"

print("Listing raw-data/ — no storage key needed, Unity Catalog handles auth:")
files = dbutils.fs.ls(f"{base_path}/raw-data/")
for f in files:
    print(f"  {f.name:<40} {f.size/1024:>8.1f} KB")

print(f"\nTotal files: {len(files)}")

In [ ]:
# ─── B4: Read a real file through the External Location ──────────────────────

customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{base_path}/raw-data/customers/customers_010626.csv")
)

print(f"customers_010626.csv — {customers_df.count():,} rows, {len(customers_df.columns)} columns")
customers_df.show(3, truncate=False)

In [ ]:
# ─── B5: Verify via SQL — confirm your External Location is registered ───────

your_external_location = "YOUR_NAME_external"   # ← e.g. virinchy_external

spark.sql("SHOW EXTERNAL LOCATIONS").show(truncate=False)
spark.sql(f"DESCRIBE EXTERNAL LOCATION {your_external_location}").show(truncate=False)

---
## Phase C — Create Your Own Lakeflow Connect Pipeline

**Goal:** Mirror the real `ecom_gbmart_conn` → `orders_data_ingestion_cdc` pipeline from ILT 2 with your own — pointing at your own Supabase project — then stand up a CDC pipeline for `orders` and `order_items`.

> **Prerequisite:** you should already have your own Supabase project with `orders` and `order_items` tables (same project pattern used in earlier Supabase hands-ons — each student runs their own). C0 below prepares them for CDC and loads the real dataset — if they're already populated from an earlier session, just verify the CDC setup below is in place.

### C0 — Set Up `orders` / `order_items` for CDC in Supabase

This mirrors exactly what was done to prepare GlobalMart's real `orders`/`order_items` tables before the production Connection existed (ILT 2, Section 3, Step 0). **Run each step below individually in the Supabase SQL Editor, and check the output, before running the next one** — do not select and run the whole script at once.

```sql
-- 1. Make sure both tables have an updated_at column
ALTER TABLE orders       ADD COLUMN IF NOT EXISTS updated_at TIMESTAMPTZ NOT NULL DEFAULT now();
ALTER TABLE order_items  ADD COLUMN IF NOT EXISTS updated_at TIMESTAMPTZ NOT NULL DEFAULT now();

-- 2. A trigger that refreshes updated_at on every INSERT/UPDATE
CREATE OR REPLACE FUNCTION set_updated_at() RETURNS trigger AS $$
BEGIN NEW.updated_at = now(); RETURN NEW; END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_orders_updated_at
BEFORE INSERT OR UPDATE ON orders
FOR EACH ROW EXECUTE FUNCTION set_updated_at();

CREATE TRIGGER trg_order_items_updated_at
BEFORE INSERT OR UPDATE ON order_items
FOR EACH ROW EXECUTE FUNCTION set_updated_at();

-- 3. REPLICA IDENTITY FULL — required for CDC to capture full row images
--    on UPDATE/DELETE, not just the primary key
ALTER TABLE orders       REPLICA IDENTITY FULL;
ALTER TABLE order_items  REPLICA IDENTITY FULL;

-- 4. A publication — exposes both tables to Lakeflow Connect over logical replication
CREATE PUBLICATION <yourname>_pub FOR TABLE orders, order_items;

-- 5. Verify both tables are in the publication
SELECT * FROM pg_publication_tables WHERE pubname = '<yourname>_pub';
```

> **Note:** `REPLICA IDENTITY FULL` and the publication are real, correct Postgres-side setup — they're what make reliable logical replication *possible* for this database in general. As C2 below shows, though, the pipeline you'll actually build is configured in **query/cursor mode**, not log-based mode, so it won't end up using this setup to stream the WAL. Do the setup anyway — it's the same real-world practice GlobalMart followed, and it keeps the door open for log-based capture later.

Then load data through **Supabase's Table Editor**, not manual `INSERT` statements — at GlobalMart's real volume (~126,000 `orders` rows, ~377,000 `order_items` rows), row-by-row inserts aren't practical:

```
Supabase → Table Editor → orders table       → Insert → Import data from CSV → upload orders.csv
Supabase → Table Editor → order_items table  → Insert → Import data from CSV → upload order_items.csv
```

Verify the row counts before moving on:
```sql
SELECT COUNT(*) FROM orders;
SELECT COUNT(*) FROM order_items;
```

### C1 — Create the Connection

```
Databricks → Catalog icon → Connections → + Add connection

  Connection name : postgresql_<yourname>          e.g. postgresql_virinchy
  Type            : PostgreSQL
  Host            : (from your Supabase project → Settings → Database)
                     pattern: aws-0-<region>.pooler.supabase.com
  Port            : 5432
  Database        : postgres
  Username        : postgres.<your_project_ref>
  Password        : (from your Supabase credentials)
  SSL             : Require
  → Test connection → Create
```

Once created, browse to your connection in Catalog Explorer and confirm `orders` and `order_items` are visible — that's your check that C0's publication is correctly exposing both tables.

### C2 — Create the Ingestion Pipeline

```
Databricks → Data Ingestion → PostgreSQL (Preview)

  Connection : postgresql_<yourname>
  Database   : postgres
  Schema     : public
  Tables     : ✅ orders
               ✅ order_items

  Destination (Unity Catalog):
    Catalog : <your_catalog>            ← your own Unity Catalog catalog
    Schema  : bronze
    Tables  : auto-created — one per source table

  Table configuration (per table):
    Primary key      : order_id / order_item_id (or whatever your own schema calls them)
    Cursor column    : updated_at         ← must increase on every change — see note below
    History tracking : SCD1 (latest state only) or SCD2 (keep every past version)
    Schedule         : skip it — do not add one       ← see note below

  → Create Pipeline → Start
```

> **Why `updated_at` and not `order_date` as the cursor column?** `order_date` is set once, when the order is placed, and never changes again. If a row is only updated later (status moves from `pending` to `shipped`), `order_date` gives the pipeline no signal that anything changed. `updated_at` is refreshed by the trigger from C0 on every INSERT and UPDATE, so it correctly sequences every change. It has no way to represent a DELETE — a deleted row has no `updated_at` left to query for.

> **History tracking:** pick SCD1 if you only need the current state of each row (matches GlobalMart's real `orders`/`order_items` pipeline) — SCD2 is the right call when the business needs every past version of a row, like tracking a customer's address history. Neither setting changes whether DELETEs are captured — that depends only on the connector mode (log-based vs. query/cursor), not on history tracking.

> **Skip the schedule.** When the pipeline wizard asks about a run schedule, leave it unset. For this lab (and for GlobalMart's real pipeline) you trigger runs manually — click **Start** whenever you want to sync a change, which is exactly what D2 and D5 below have you do. A production deployment would eventually add a periodic schedule (e.g. hourly), but that's a separate, later decision — not something to configure here.

> This pipeline's cursor column is `updated_at`, so each run queries `WHERE updated_at > last_seen_value` and upserts whatever comes back — that's **query/cursor-based capture**, not WAL streaming. `REPLICA IDENTITY FULL` and the C0 publication are correct Postgres-side setup for logical replication in general, but this pipeline doesn't use them to stream the WAL. As a result it captures every INSERT and UPDATE, but — like any cursor/query-based pipeline — it **cannot detect hard DELETEs**. Phase D proves both halves of that.

### C3 — Record the first-run result

```
Status    : Completed?  ___________
orders      Upserted:   ___________
order_items Upserted:   ___________
Duration    :           ___________
```

In [ ]:
# ─── C4: Verify the pipeline landed in Unity Catalog ──────────────────────────
# Replace with your own catalog name from Phase C2

your_catalog = "YOUR_CATALOG_NAME"   # ← the catalog you picked as pipeline destination

orders_count = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.orders").collect()[0]["n"]
items_count  = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.order_items").collect()[0]["n"]

print(f"orders      : {orders_count:,} rows")
print(f"order_items : {items_count:,} rows")

spark.sql(f"DESCRIBE EXTENDED {your_catalog}.bronze.orders").show(truncate=False)
# Look for: Type = STREAMING_TABLE, Catalog Name = your_catalog (not hive_metastore)

---
## Phase D — Prove CDC, and See the Real Limit

**Goal:** Make a change in Supabase, re-run the pipeline, and show that only the change syncs — not the whole table. Then see the real limit of cursor-based capture for yourself: because your pipeline queries `WHERE updated_at > last_seen_value` on each run (not the WAL), a hard DELETE does **not** sync.

> `REPLICA IDENTITY FULL` and the C0 publication are real, correct Postgres-side setup — they're what make reliable logical replication possible in general — but this pipeline's cursor/query mode doesn't use them to stream the WAL. Phase D's bonus step (D5) proves the DELETE gap directly instead of just asserting it.

### D1 — Make a change in Supabase SQL Editor

```sql
-- Insert 1 new order
INSERT INTO orders (order_id, customer_id, order_date, status, payment_method_id, shipping_tier_id)
VALUES ('O-TEST-001', 'CUST-99999', NOW(), 'pending', 'PM-001', 'STD');

-- Update it — simulate it shipping
UPDATE orders SET status = 'shipped' WHERE order_id = 'O-TEST-001';
```

### D2 — Re-run the pipeline

```
Databricks → Jobs & Pipelines → postgresql_<yourname> → Start

Expected result:
  Status   : Completed
  Upserted : 1        ← only the 1 changed row, not the whole orders table
  Duration : faster than the first run
```

### D3 — Confirm the row in Unity Catalog

In [ ]:
# ─── D4: Find the test row — prove it landed via CDC, not a full reload ───────

result_df = spark.sql(f"""
    SELECT order_id, customer_id, status
    FROM {your_catalog}.bronze.orders
    WHERE order_id = 'O-TEST-001'
""")
result_df.show(truncate=False)

new_orders_count = spark.sql(f"SELECT COUNT(*) AS n FROM {your_catalog}.bronze.orders").collect()[0]["n"]
print(f"orders row count now: {new_orders_count:,}  (was {orders_count:,} before D1 — should be +1)")
print("status should read 'shipped' — the UPDATE was captured, not just the INSERT")

---
### D5 (Bonus) — Prove the DELETE Is NOT Captured

Now delete the test row entirely in Supabase — and check whether it disappears from Bronze. (Spoiler: it won't. That's the point — this is the real limit of cursor-based capture.)

```sql
-- In Supabase SQL Editor:
DELETE FROM orders WHERE order_id = 'O-TEST-001';
```

Re-run your pipeline (same as D2), then run the verification cell below.

In [ ]:
# ─── D6: Verify the DELETE did NOT propagate — this pipeline is cursor/query-based, not WAL streaming ──
# Each run queries WHERE updated_at > last_seen_value. A deleted row has nothing left to match —
# it isn't "updated", it's just gone from the source — so Lakeflow Connect never sees a DELETE
# event and never removes the row from Bronze. This is true whether you configured SCD1 or SCD2
# history tracking in C2: neither one can capture a change the pipeline never observed.
# REPLICA IDENTITY FULL and the C0 publication remain correct setup for logical replication in
# general — this pipeline just doesn't use them, because it isn't reading the WAL.

still_there_df = spark.sql(f"""
    SELECT order_id, customer_id, status
    FROM {your_catalog}.bronze.orders
    WHERE order_id = 'O-TEST-001'
""")

row_count = still_there_df.count()
if row_count > 0:
    print("As expected: the deleted row is STILL in Bronze — cursor-based capture cannot see a DELETE.")
    still_there_df.show(truncate=False)
else:
    print("Unexpected — the row is gone. Double-check your pipeline is configured with a cursor")
    print("column (updated_at) in query/cursor mode, not a log-based/WAL connector mode, and")
    print("re-run this cell after the pipeline has finished its manual run.")

---
## Submission Checklist

> ⚠️ **Replace any real password/secret values with placeholders before uploading this notebook.**

```
Submission Checklist
────────────────────────────────────────────────────────────────
✅ Access Connector confirmed / created, Resource ID copied
✅ Access Connector has Storage Blob Data Contributor on my storage account
✅ Storage Credential created: <yourname>_credential (status: Active)
✅ External Location created: <yourname>_external
✅ "Test connection" succeeded on the External Location
✅ dbutils.fs.ls() listed raw/ files — no storage key used
✅ customers CSV read successfully through the External Location
✅ SHOW EXTERNAL LOCATIONS / DESCRIBE EXTERNAL LOCATION confirmed
✅ C0 setup script run in Supabase SQL Editor: updated_at trigger, REPLICA IDENTITY FULL,
   and publication created + verified (pg_publication_tables shows both tables)
✅ orders / order_items data loaded via Table Editor → Import data from CSV (not manual INSERTs)
── orders row count after import:        ______
── order_items row count after import:   ______
✅ PostgreSQL Connection created: postgresql_<yourname>
✅ Ingestion Pipeline created (query/cursor-based capture; cursor column: updated_at, history
   tracking: SCD1, no schedule — triggered manually) and first run completed
── orders row count (first run):        ______
── order_items row count (first run):   ______
✅ CDC proof: inserted + updated 1 row in Supabase
✅ Re-ran pipeline — confirmed Upserted: 1 (not a full reload)
── orders row count (after CDC run):     ______
✅ Bonus: deleted the test row in Supabase, re-ran pipeline, confirmed it's STILL PRESENT in
   Bronze — proving cursor-based capture cannot detect hard deletes
✅ Screenshot: Storage Credential + External Location pages
✅ Screenshot: Ingestion Pipeline run history (all runs, including the DELETE test)
✅ Notebook uploaded with all secrets replaced by placeholders
────────────────────────────────────────────────────────────────
```